<a href="https://colab.research.google.com/github/D-Barradas/Accelerated-Data-Science-with-RAPIDS/blob/main/part4/04_1_RAPIDS_PyTorch_RayTune_Tabular_HPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAPIDS + PyTorch + Ray Tune: GPU-Accelerated Tabular HPO

This tutorial walks through a full GPU-first tabular machine learning workflow:

RAPIDS cuDF preprocessing -> CuPy arrays -> PyTorch neural network -> Ray Tune hyperparameter optimization with ASHA

## Learning objectives
By the end of this notebook, you will be able to:
- Create and process tabular data using cuDF
- Engineer and normalize features on GPU
- Convert RAPIDS/CuPy data into PyTorch tensors
- Train a PyTorch MLP classifier
- Optimize hyperparameters with Ray Tune
- Use ASHA early stopping
- Retrieve and evaluate the best model configuration

## 1. Colab setup

Use a **GPU runtime** in Google Colab before continuing: Runtime -> Change runtime type -> Hardware accelerator -> GPU.

> If RAPIDS installation fails, check the current RAPIDS Colab installation instructions at https://rapids.ai/.

In [ ]:
# Check the active GPU
!nvidia-smi

In [ ]:
# RAPIDS installation in Colab is version-sensitive.
# This command uses NVIDIA's Colab bootstrap script, which handles CUDA compatibility.
# If this fails, restart runtime and consult https://rapids.ai/.
!bash <(curl -s https://raw.githubusercontent.com/rapidsai/rapidsai-csp-utils/main/colab/rapids-colab.sh)

In [ ]:
# Install Ray Tune, gdown (dataset download), and lightweight dependencies used in this notebook
!pip install -q "ray[tune]" scikit-learn matplotlib pandas gdown

## 2. Imports and configuration

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cudf
import cupy as cp

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, roc_auc_score

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
cp.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using torch device: {device}")
print(f"Torch version: {torch.__version__}")
print(f"Ray version: {ray.__version__}")
print(f"cuDF version: {cudf.__version__}")
print(f"CuPy version: {cp.__version__}")

## 3. Data ingestion: airline on-time dataset (ORC) from Google Drive

Instead of a synthetic dataset, we now load a real ~2003 airline on-time performance file stored as Apache ORC. We mount Google Drive so the (multi-hundred-MB) file persists across Colab sessions, download it once with `gdown` if it isn't already there, then load it directly onto the GPU with `cudf.read_orc`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gdown

dir_path = '/content/drive/MyDrive/Accel_DS_RAPIDS'
file_path = 'part4/data/airline-data-full-2003.orc'
absolute_path_to_file = os.path.join(dir_path, file_path)

print(f"Checking existence of directory: {os.path.dirname(absolute_path_to_file)}")
os.makedirs(os.path.dirname(absolute_path_to_file), exist_ok=True)

print(f"\nChecking existence of file: {absolute_path_to_file}")
if os.path.exists(absolute_path_to_file):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' not found.")
    print(f"Downloading...")
    url = "https://drive.google.com/uc?id=1EecyIkVpGMWBm3AHGfCQij4YN92Cjddf"
    gdown.download(url, absolute_path_to_file, quiet=False)

In [ ]:
NUMERIC_FEATURES = ["Month", "DayofMonth", "DayOfWeek", "CRSDepTime", "CRSArrTime", "Distance"]
CATEGORICAL_FEATURES = ["UniqueCarrier", "Origin", "Dest"]
DELAY_THRESHOLD_MIN = 15

raw_gdf = cudf.read_orc(absolute_path_to_file)
print("Raw shape:", raw_gdf.shape)

# Cancelled/diverted flights have no meaningful arrival delay; drop them along with missing fields.
raw_gdf = raw_gdf[raw_gdf["Cancelled"] == 0]
if "Diverted" in raw_gdf.columns:
    raw_gdf = raw_gdf[raw_gdf["Diverted"] == 0]
required_cols = ["ArrDelay"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES
raw_gdf = raw_gdf.dropna(subset=required_cols).reset_index(drop=True)

df = raw_gdf[NUMERIC_FEATURES].astype("float32")
for col in CATEGORICAL_FEATURES:
    df[col] = raw_gdf[col].astype("category").cat.codes.astype("float32")

# Binary target: did the flight arrive more than 15 minutes late?
df["target"] = (raw_gdf["ArrDelay"] > DELAY_THRESHOLD_MIN).astype(np.int32)

print("Shape:", df.shape)
display(df.head())

class_balance = df["target"].value_counts().sort_index()
print("\nClass balance:")
display(class_balance)

## 4. RAPIDS GPU data processing

All operations below run on GPU with cuDF: a missing-value check, one engineered feature (hour-of-day extracted from the scheduled departure time), and z-score normalization of every feature column.

In [ ]:
# 1) Missing value inspection
missing_counts = df.isnull().sum()
total_missing = int(missing_counts.sum())
print(f"Total missing values: {total_missing}")

# 2) Feature engineering on GPU: extract hour-of-day from the scheduled departure time (HHMM format)
eps = 1e-6
df["dep_hour"] = (df["CRSDepTime"] // 100).astype("float32")

# 3) Normalize all feature columns on GPU
all_feature_cols = [c for c in df.columns if c != "target"]
for col in all_feature_cols:
    col_mean = df[col].mean()
    col_std = df[col].std()
    df[col] = (df[col] - col_mean) / (col_std + eps)

print(f"Total features after engineering: {len(all_feature_cols)}")
display(df.head())

## 5. Train/validation/test split and tensor conversion

We split data with CuPy index permutations on GPU, then convert to formats needed for PyTorch and Ray Tune.

In [ ]:
X_cudf = df[all_feature_cols]
y_cudf = df["target"]

X_cp = X_cudf.to_cupy()
y_cp = y_cudf.to_cupy().astype(cp.float32)

n = X_cp.shape[0]
perm = cp.random.permutation(n)

train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_idx = perm[:train_end]
val_idx = perm[train_end:val_end]
test_idx = perm[val_end:]

X_train_cp, y_train_cp = X_cp[train_idx], y_cp[train_idx]
X_val_cp, y_val_cp = X_cp[val_idx], y_cp[val_idx]
X_test_cp, y_test_cp = X_cp[test_idx], y_cp[test_idx]

print("Split sizes:")
print("Train:", X_train_cp.shape[0])
print("Val:  ", X_val_cp.shape[0])
print("Test: ", X_test_cp.shape[0])

In [ ]:
def cupy_to_torch_tensor(x_cp):
    """Zero-copy CuPy -> PyTorch conversion via DLPack; falls back to NumPy if unsupported."""
    try:
        return torch.utils.dlpack.from_dlpack(x_cp.toDlpack())
    except Exception:
        return torch.from_numpy(cp.asnumpy(x_cp))


def cupy_split_to_numpy(x_cp, y_cp):
    """Convert one CuPy (X, y) split to NumPy via a zero-copy DLPack GPU tensor.

    Ray Tune trials run as separate worker processes, so a live CUDA tensor from this
    driver process can't be handed to them directly -- each trial rebuilds its own CUDA
    tensors from the returned NumPy arrays once it starts on its assigned GPU fraction.
    Routing through `cupy_to_torch_tensor` still avoids an extra CuPy-side copy: only the
    final GPU tensor -> NumPy step touches host memory.
    """
    x_t = cupy_to_torch_tensor(x_cp).float()
    y_t = cupy_to_torch_tensor(y_cp).float().view(-1, 1)
    return x_t.cpu().numpy(), y_t.cpu().numpy()


X_train_np, y_train_np = cupy_split_to_numpy(X_train_cp, y_train_cp)
X_val_np, y_val_np = cupy_split_to_numpy(X_val_cp, y_val_cp)
X_test_np, y_test_np = cupy_split_to_numpy(X_test_cp, y_test_cp)

print("Train tensor shapes:", X_train_np.shape, y_train_np.shape)

## 6. PyTorch Dataset and DataLoader

In [ ]:
train_ds = TensorDataset(
    torch.from_numpy(X_train_np),
    torch.from_numpy(y_train_np),
)

val_ds = TensorDataset(
    torch.from_numpy(X_val_np),
    torch.from_numpy(y_val_np),
)

test_ds = TensorDataset(
    torch.from_numpy(X_test_np),
    torch.from_numpy(y_test_np),
)

def make_dataloaders(batch_size):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return train_loader, val_loader, test_loader

## 7. Define a PyTorch MLP model

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        layers = []
        in_dim = input_dim

        for _ in range(num_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 1))  # single logit for binary classification
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

criterion = nn.BCEWithLogitsLoss()

## 8. Training and evaluation helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float()

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    probs_all = []
    y_all = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float()

        logits = model(xb)
        loss = criterion(logits, yb)
        running_loss += loss.item() * xb.size(0)

        probs = torch.sigmoid(logits)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    probs_all = np.vstack(probs_all).reshape(-1)
    y_all = np.vstack(y_all).reshape(-1)
    preds = (probs_all >= 0.5).astype(np.int32)

    loss_mean = running_loss / len(loader.dataset)
    acc = accuracy_score(y_all, preds)

    try:
        auc = roc_auc_score(y_all, probs_all)
    except Exception:
        auc = float("nan")

    return {"loss": loss_mean, "accuracy": acc, "auc": auc}

## 9. Define the Ray Tune training function

In Colab, you usually have one GPU. Ray Tune can still run multiple trials, often sequentially on that single GPU, while ASHA prunes weak configurations early.

In [ ]:
INPUT_DIM = X_train_np.shape[1]
MAX_EPOCHS = 10

def train_tune(config):
    train_loader, val_loader, _ = make_dataloaders(config["batch_size"])

    model = TabularMLP(
        input_dim=INPUT_DIM,
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)

        tune.report(
            epoch=epoch,
            train_loss=train_loss,
            val_loss=val_metrics["loss"],
            val_accuracy=val_metrics["accuracy"],
            val_auc=val_metrics["auc"],
        )

## 10. Configure search space and ASHA scheduler

ASHA (Asynchronous Successive Halving Algorithm) quickly stops underperforming trials, so compute is focused on promising configurations.

In [ ]:
search_space = {
    "hidden_dim": tune.choice([64, 128, 256]),
    "num_layers": tune.choice([1, 2, 3]),
    "dropout": tune.uniform(0.0, 0.5),
    "lr": tune.loguniform(1e-4, 1e-2),
    "weight_decay": tune.loguniform(1e-6, 1e-3),
    "batch_size": tune.choice([256, 512, 1024]),
}

asha = ASHAScheduler(
    metric="val_loss",
    mode="min",
    max_t=10,
    grace_period=2,
    reduction_factor=2,
)

## 11. Run HPO with Ray Tune

In [ ]:
if ray.is_initialized():
    ray.shutdown()

ray.init(ignore_reinit_error=True, log_to_driver=False)
print("Ray initialized.")

num_samples = 8  # Increase later for a deeper search
gpu_per_trial = 1 if torch.cuda.is_available() else 0

trainable = tune.with_resources(train_tune, {"cpu": 2, "gpu": gpu_per_trial})

# Prefer modern Ray Tune API (Tuner), then fallback to tune.run for compatibility.
try:
    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            metric="val_loss",
            mode="min",
            scheduler=asha,
            num_samples=num_samples,
        ),
        run_config=ray.air.RunConfig(name="rapids_pytorch_raytune_tabular_hpo"),
    )
    results = tuner.fit()
except Exception as e:
    print(f"Tuner API failed ({e}); falling back to tune.run.")
    analysis = tune.run(
        trainable,
        config=search_space,
        metric="val_loss",
        mode="min",
        scheduler=asha,
        num_samples=num_samples,
        resources_per_trial={"cpu": 2, "gpu": gpu_per_trial},
        name="rapids_pytorch_raytune_tabular_hpo",
    )
    class _TuneRunWrapper:
        def __init__(self, analysis_obj):
            self.analysis_obj = analysis_obj
        def get_best_result(self, metric, mode):
            best_cfg = self.analysis_obj.get_best_config(metric=metric, mode=mode)
            best_trial = self.analysis_obj.get_best_trial(metric=metric, mode=mode, scope="last")
            class _Best:
                config = best_cfg
                metrics = best_trial.last_result
            return _Best()
        def get_dataframe(self):
            return self.analysis_obj.dataframe()
    results = _TuneRunWrapper(analysis)

In [ ]:
best_result = results.get_best_result(metric="val_loss", mode="min")
best_config = best_result.config

print("Best config:")
print(best_config)
print("Best validation loss:", best_result.metrics.get("val_loss"))

results_df = results.get_dataframe()
show_cols = [
    "trial_id", "val_loss", "val_accuracy", "val_auc",
    "config/hidden_dim", "config/num_layers", "config/dropout",
    "config/lr", "config/weight_decay", "config/batch_size"
]
show_cols = [c for c in show_cols if c in results_df.columns]
display(results_df[show_cols].sort_values(by="val_loss").head(10))

## 12. Train final model with best hyperparameters

In [ ]:
# Combine train + validation for final fitting
X_trainval_np = np.vstack([X_train_np, X_val_np]).astype(np.float32)
y_trainval_np = np.vstack([y_train_np, y_val_np]).astype(np.float32)

trainval_ds = TensorDataset(
    torch.from_numpy(X_trainval_np),
    torch.from_numpy(y_trainval_np),
)

final_batch_size = best_config["batch_size"]
trainval_loader = DataLoader(trainval_ds, batch_size=final_batch_size, shuffle=True, num_workers=0, pin_memory=True)
_, _, test_loader = make_dataloaders(final_batch_size)

final_model = TabularMLP(
    input_dim=INPUT_DIM,
    hidden_dim=best_config["hidden_dim"],
    num_layers=best_config["num_layers"],
    dropout=best_config["dropout"],
).to(device)

final_optimizer = optim.Adam(
    final_model.parameters(),
    lr=best_config["lr"],
    weight_decay=best_config["weight_decay"],
)

FINAL_EPOCHS = 8
history = []
for epoch in range(1, FINAL_EPOCHS + 1):
    train_loss = train_one_epoch(final_model, trainval_loader, final_optimizer, device)
    test_metrics = evaluate(final_model, test_loader, device)
    history.append((epoch, train_loss, test_metrics["loss"], test_metrics["accuracy"], test_metrics["auc"]))
    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
        f"test_loss={test_metrics['loss']:.4f} | test_acc={test_metrics['accuracy']:.4f} | test_auc={test_metrics['auc']:.4f}"
    )

In [ ]:
final_test_metrics = evaluate(final_model, test_loader, device)
print("\nFinal held-out test metrics:")
print(f"Test loss:     {final_test_metrics['loss']:.4f}")
print(f"Test accuracy: {final_test_metrics['accuracy']:.4f}")
print(f"Test ROC AUC:  {final_test_metrics['auc']:.4f}")

## 13. Visualize HPO results

In [ ]:
plot_df = results_df.copy()
if "trial_id" not in plot_df.columns:
    plot_df["trial_id"] = np.arange(len(plot_df)).astype(str)

plot_df = plot_df.sort_values("val_loss").head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(plot_df["trial_id"].astype(str), plot_df["val_loss"])
axes[0].set_title("Validation Loss by Trial")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Val Loss")
axes[0].tick_params(axis="x", rotation=70)

axes[1].bar(plot_df["trial_id"].astype(str), plot_df["val_accuracy"])
axes[1].set_title("Validation Accuracy by Trial")
axes[1].set_xlabel("Trial")
axes[1].set_ylabel("Val Accuracy")
axes[1].tick_params(axis="x", rotation=70)

plt.tight_layout()
plt.show()

In [ ]:
# Optional: inspect the distribution of selected learning rates
if "config/lr" in results_df.columns:
    plt.figure(figsize=(6, 4))
    plt.hist(results_df["config/lr"], bins=10)
    plt.title("Distribution of Sampled Learning Rates")
    plt.xlabel("Learning rate")
    plt.ylabel("Count")
    plt.show()

## 14. Student exercises

1. Increase `num_samples` from 8 to 20 and compare the best validation loss and test accuracy.
2. Add one more engineered feature in cuDF (for example, `f6 / (abs(f7) + eps)`).
3. Add Batch Normalization to `TabularMLP`, and tune whether to enable it.
4. Change the optimization objective from `val_loss` to `val_accuracy`.
5. Increase dataset size (for example, 300,000 rows) and measure runtime changes.
6. Replace the synthetic dataset with a CSV loaded using `cudf.read_csv(...)`.

## 15. Summary

You built an end-to-end GPU workflow for tabular deep learning and HPO:

- RAPIDS/cuDF accelerated dataframe preprocessing and feature engineering.
- PyTorch trained a neural network for binary classification on GPU.
- Ray Tune automated hyperparameter optimization.
- ASHA improved efficiency by pruning weak trials early.

This pattern scales naturally to larger tabular datasets and production-style experimentation loops.